# Week 12 Lab — Discretization, boundaries, and unconfined flow

**HWRS 564a · Fall 2026**

Last week's model converged on the first try. That was not luck — it was because
the aquifer was **confined**, which makes the equations linear.

This week you take the confining layer off. The physics gets more realistic and
the model stops converging, and dealing with that is most of what a modeller
actually does. On the way we look at how much your answer depends on the grid you
chose, and at the three things `ibound` can mean.

## How to use this notebook

Run each cell with **Shift+Enter**. Cells marked  **`# YOUR TURN`**  have
something for you to write. Cells marked **`# CHECK`** verify your answer — if
they run without complaint, you're right.

> **Before you submit anything all semester:** *Kernel → Restart Kernel and Run
> All Cells*. A notebook that only works when run out of order is not finished.


## Learning objectives

By the end of this notebook you can:

1. Test whether an answer depends on cell size, and say what that means
2. Use `ibound = 0` to make a domain that isn't a rectangle
3. Explain the difference between `laytyp=0` and `laytyp=1`, and why it matters
4. Diagnose a non-convergent model from the `.list` file
5. Fix one by improving the initial condition and relaxing the solver
6. Recognise dry cells and say why they are a modelling failure, not a result

---

## Part 1 — Setup

In [ ]:
from pathlib import Path

import flopy
import matplotlib.pyplot as plt
import numpy as np

ROOT = next(p for p in Path.cwd().parents if (p / "pyproject.toml").exists())
DATA = ROOT / "data"
MF_EXE = ROOT / "modflow" / "mf2005"
assert MF_EXE.exists(), (
    f"MODFLOW binary not found at {MF_EXE}. Run ./postbuild.sh from a terminal."
)

RUN = ROOT / "_run"


def discrepancy(list_path):
    """Percent mass-balance discrepancy from a MODFLOW .list file."""
    for line in reversed(Path(list_path).read_text().splitlines()):
        if "PERCENT DISCREPANCY" in line:
            return float(line.split()[-1])
    raise ValueError(f"no budget discrepancy in {list_path}")


print("ready")

---

## Part 2 — Does the answer depend on the grid?

You chose 250 m cells last week because I told you to. **A number that changes
when you refine the grid is a number about your grid, not about the aquifer.**

The test is simple: run the same physical problem at two resolutions and see
whether the answer moves.

In [ ]:
def confined_basin(cell_size, well_q=-20000.0, name="grid"):
    """The Week 11 model at an arbitrary cell size. Returns (model, head).

    The domain stays 15 km x 10 km, so a smaller cell means more cells. The
    well is placed at the same *physical location*, not the same cell index.
    """
    ncol = int(15000 / cell_size)
    nrow = int(10000 / cell_size)
    ws = RUN / f"week12_{name}_{int(cell_size)}"
    ws.mkdir(parents=True, exist_ok=True)

    mf = flopy.modflow.Modflow("basin", model_ws=str(ws), exe_name=str(MF_EXE))
    flopy.modflow.ModflowDis(mf, 1, nrow, ncol, delr=cell_size, delc=cell_size,
                             top=600.0, botm=400.0, nper=1, steady=True)

    ibound = np.ones((1, nrow, ncol), dtype=int)
    ibound[:, :, 0] = -1
    ibound[:, :, -1] = -1
    strt = np.full((1, nrow, ncol), 660.0)
    strt[:, :, 0] = 620.0
    strt[:, :, -1] = 700.0
    flopy.modflow.ModflowBas(mf, ibound=ibound, strt=strt)

    flopy.modflow.ModflowLpf(mf, hk=12.0, laytyp=0, ipakcb=53)

    # the wellfield sits 7500 m east, 5000 m south of the north-west corner
    wr, wc = int(5000 / cell_size), int(7500 / cell_size)
    flopy.modflow.ModflowWel(mf, stress_period_data={0: [[0, wr, wc, well_q]]},
                             ipakcb=53)
    flopy.modflow.ModflowRch(mf, rech=1.2e-4, ipakcb=53)
    flopy.modflow.ModflowPcg(mf)
    flopy.modflow.ModflowOc(mf, stress_period_data={(0, 0): ["save head", "save budget"]})

    mf.write_input()
    ok, buff = mf.run_model(silent=True, report=True)
    assert ok, f"{cell_size} m grid failed:\n" + "\n".join(buff[-15:])
    head = flopy.utils.HeadFile(str(ws / "basin.hds")).get_data()
    return mf, head, (wr, wc)


results = {}
for size in [500.0, 250.0, 125.0]:
    mf, head, (wr, wc) = confined_basin(size)
    results[size] = {
        "n_cells": head[0].size,
        "well_head": float(head[0, wr, wc]),
        "min_head": float(head[0].min()),
        "discrepancy": discrepancy(Path(mf.model_ws) / "basin.list"),
    }
    print(f"{size:6.0f} m  {results[size]['n_cells']:7,d} cells  "
          f"well head {results[size]['well_head']:8.3f} m  "
          f"discrepancy {results[size]['discrepancy']:+.3f} %")

### YOUR TURN 1

The well head moves as the grid refines. Quantify it.

- `head_500`, `head_250`, `head_125` — the well-cell head at each resolution
- `change_500_to_250` — how much the answer moved when you halved the cell size
- `change_250_to_125` — and again

A converging sequence has each change *smaller* than the last. If the changes
are not shrinking, the grid is not fine enough to be saying anything.

In [ ]:
# YOUR TURN
head_500 = ...
head_250 = ...
head_125 = ...
change_500_to_250 = ...
change_250_to_125 = ...

In [ ]:
# CHECK
assert abs(head_250 - 654.36) < 0.2, f"the 250 m answer should match Week 11: got {head_250}"
assert abs(change_500_to_250) > abs(change_250_to_125), (
    "the changes should be shrinking — that is what grid convergence looks like"
)
print(f"500 m -> 250 m : the well head moved {change_500_to_250:+7.3f} m")
print(f"250 m -> 125 m : the well head moved {change_250_to_125:+7.3f} m")
print(f"\nratio: {abs(change_500_to_250 / change_250_to_125):.1f}x smaller. Correct.")

**The drawdown at a pumping cell never fully converges, and that is expected.**

Head near a well varies logarithmically with distance, so the *average* head over
a cell containing a well depends on how big that cell is — forever. Refine the
grid and the well cell gets deeper, without limit.

What that means in practice:

- **Regional heads and the water budget do converge.** Those are the numbers you
  report.
- **The head in a pumping cell is not a water level in a well.** If you need
  that, you apply an analytical correction, or you refine locally, or you say
  you can't.

Reporting a pumping-cell head as a predicted well water level is one of the most
common errors in groundwater modelling reports.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

sizes = sorted(results, reverse=True)
axes[0].plot([results[s]["n_cells"] for s in sizes],
             [results[s]["well_head"] for s in sizes],
             "o-", color="#AB0520", ms=6)
axes[0].set_xscale("log")
axes[0].set_xlabel("number of cells")
axes[0].set_ylabel("head in the well cell (m)")
axes[0].set_title("The pumping cell keeps deepening")
axes[0].grid(alpha=0.3)

axes[1].plot([results[s]["n_cells"] for s in sizes],
             [results[s]["min_head"] for s in sizes],
             "s-", color="#0C234B", ms=6)
axes[1].set_xscale("log")
axes[1].set_xlabel("number of cells")
axes[1].set_ylabel("minimum head in the domain (m)")
axes[1].set_title("The boundary head does not move at all")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## Part 3 — `ibound`, and domains that aren't rectangles

`ibound` takes three values and they mean three quite different things.

| Value | Meaning | Head is |
|---|---|---|
| `1` | active | solved for |
| `0` | **inactive** — not part of the model | not computed at all |
| `-1` | constant head | fixed at `strt`, and never changes |

Real basins are not rectangles. `ibound = 0` is how you cut the shape out.

In [ ]:
NROW, NCOL, DELR = 40, 60, 250.0
land_surface = np.loadtxt(DATA / "tucson_grid_top.csv", delimiter=",")

# Mark the mountain-front cells inactive: bedrock above 850 m, not basin fill
ibound = np.ones((1, NROW, NCOL), dtype=int)
ibound[0][land_surface > 850.0] = 0
ibound[:, :, 0] = -1
ibound[:, :, -1] = -1

n_inactive = int((ibound == 0).sum())
print(f"{n_inactive} cells marked inactive ({100 * n_inactive / ibound.size:.0f}% of the grid)")
print(f"{int((ibound == 1).sum())} active, {int((ibound == -1).sum())} constant head")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
cmap = plt.matplotlib.colors.ListedColormap(["#AB0520", "#E2E9EB", "#1E5288"])
im = ax.imshow(ibound[0], cmap=cmap, vmin=-1.5, vmax=1.5,
               extent=[0, NCOL * DELR / 1000, 0, NROW * DELR / 1000])
cb = fig.colorbar(im, ax=ax, ticks=[-1, 0, 1], shrink=0.85)
cb.ax.set_yticklabels(["-1 constant head", "0 inactive", "1 active"])
ax.set_xlabel("distance east (km)")
ax.set_ylabel("distance north (km)")
ax.set_title("The basin cut out of the rectangle: bedrock above 850 m is inactive")
plt.tight_layout()
plt.show()

> **`ibound = 0` is not the same as `hk = 0`.** An inactive cell is removed from
> the equations entirely — no head is computed, and it appears as `1e30` in the
> output. A cell with zero conductivity is still solved for, still has a head,
> and still contributes a row to the matrix. The first is a statement about the
> domain; the second is a statement about the geology, and it will make your
> solver miserable.

### YOUR TURN 2

Inactive cells break flow paths, and MODFLOW will not warn you. Check whether
cutting out the mountain front has isolated any part of the basin.

For each **row**, count the active cells, and find whether any row has become
entirely inactive.

- `active_per_row` — number of `ibound == 1` cells in each row (length 40)
- `fully_blocked_rows` — how many rows have zero active cells

In [ ]:
# YOUR TURN
active_per_row = ...
fully_blocked_rows = ...

In [ ]:
# CHECK
assert active_per_row.shape == (NROW,), f"expected 40 values, got {active_per_row.shape}"
assert fully_blocked_rows == 3, (
    f"expected to find 3 blocked rows with an 850 m cutoff, got {fully_blocked_rows}"
)
print(f"active cells per row: min {active_per_row.min()}, max {active_per_row.max()}")
print(f"fully blocked rows:   {fully_blocked_rows}")
print(f"which rows:           {np.where(active_per_row == 0)[0].tolist()}")
print("\nFound them. Correct — and that is a problem, not a result.")

**The 850 m cutoff isolated the three southernmost rows.** Land surface rises
above 850 m across the entire southern edge, so those rows became a wall with
nothing behind it.

MODFLOW would have run this happily. It would have solved the remaining domain,
computed nothing for rows 37–39, and reported a closed budget — because a region
that exchanges no water with anything is trivially in balance.

**Checking connectivity is your job, and nothing in the software does it for
you.**

### YOUR TURN 3

Fix it. Find the lowest cutoff elevation, in 10 m steps from 850 m upward, that
leaves every row with at least one active cell.

- `working_cutoff` — the lowest such elevation
- `n_inactive_at_cutoff` — how many cells are inactive at that cutoff

In [ ]:
# YOUR TURN
working_cutoff = ...
n_inactive_at_cutoff = ...

In [ ]:
# CHECK
assert 850 < working_cutoff <= 900, f"got {working_cutoff}"
check = np.ones((NROW, NCOL), dtype=int)
check[land_surface > working_cutoff] = 0
assert ((check == 1).sum(axis=1) > 0).all(), "that cutoff still blocks a row"
lower = np.ones((NROW, NCOL), dtype=int)
lower[land_surface > working_cutoff - 10] = 0
assert not ((lower == 1).sum(axis=1) > 0).all(), (
    f"{working_cutoff - 10} m also works, so it isn't the lowest"
)
print(f"lowest workable cutoff: {working_cutoff:.0f} m")
print(f"inactive cells there:   {n_inactive_at_cutoff}")
print("Correct.")

> **Notice what just happened to the model.** We changed a *geological* decision
> — where basin fill ends and bedrock begins — because of a *numerical* problem.
> That is sometimes the right call and sometimes the beginning of a model that
> answers a different question than the one you asked.
>
> Either way, write it down. "The bedrock cutoff was raised from 850 m to 890 m
> to maintain hydraulic connectivity along the southern boundary" belongs in the
> report, not in your head.

In [ ]:
# rebuild ibound with the workable cutoff
ibound = np.ones((1, NROW, NCOL), dtype=int)
ibound[0][land_surface > working_cutoff] = 0
ibound[:, :, 0] = -1
ibound[:, :, -1] = -1
print(f"{int((ibound == 0).sum())} inactive, {int((ibound == 1).sum())} active, "
      f"{int((ibound == -1).sum())} constant head")

---

## Part 4 — Taking the confining layer off

Everything so far used `laytyp=0`, confined:

$$T = K b \quad \text{with } b \text{ fixed}$$

Transmissivity is a constant, the equations are linear, and one pass of the
solver gets the answer.

`laytyp=1` means **convertible**, which for a water-table aquifer means:

$$T = K (h - z_{bot})$$

Transmissivity now depends on the head you are solving for. The equation is
**nonlinear**, and MODFLOW has to iterate: guess heads, compute $T$, solve,
recompute $T$, solve again.

In [ ]:
def unconfined_basin(strt_value, pcg_kwargs, name):
    """Water-table version of the basin. Returns (success, buff, model)."""
    ws = RUN / f"week12_{name}"
    ws.mkdir(parents=True, exist_ok=True)

    mf = flopy.modflow.Modflow("uncf", model_ws=str(ws), exe_name=str(MF_EXE))
    flopy.modflow.ModflowDis(mf, 1, NROW, NCOL, delr=DELR, delc=DELR,
                             top=700.0, botm=400.0, nper=1, steady=True)

    ib = np.ones((1, NROW, NCOL), dtype=int)
    ib[:, :, 0] = -1
    ib[:, :, -1] = -1
    strt = np.full((1, NROW, NCOL), strt_value)
    strt[:, :, 0] = 620.0
    strt[:, :, -1] = 680.0
    flopy.modflow.ModflowBas(mf, ibound=ib, strt=strt)

    flopy.modflow.ModflowLpf(mf, hk=12.0, laytyp=1, ipakcb=53)   # <- convertible
    flopy.modflow.ModflowWel(mf, stress_period_data={0: [[0, 20, 30, -20000.0]]},
                             ipakcb=53)
    flopy.modflow.ModflowRch(mf, rech=1.2e-4, ipakcb=53)
    flopy.modflow.ModflowPcg(mf, **pcg_kwargs)
    flopy.modflow.ModflowOc(mf, stress_period_data={(0, 0): ["save head", "save budget"]})

    mf.write_input()
    ok, buff = mf.run_model(silent=True, report=True)
    return ok, buff, mf


ok_default, buff_default, mf_default = unconfined_basin(660.0, {}, "uncf_default")
print(f"default solver settings: success = {ok_default}")
for line in [l.strip() for l in buff_default if l.strip()][-3:]:
    print(f"  {line}")

**It fails.** Exactly the same physical problem as last week, one argument
different, and the solver cannot get there.

This is the normal condition of groundwater modelling. Read the diagnosis:

In [ ]:
listing = (Path(mf_default.model_ws) / "uncf.list").read_text().splitlines()

for line in [l.strip() for l in listing if l.strip()][-8:]:
    print(line)

print(f"\npercent discrepancy: {discrepancy(Path(mf_default.model_ws) / 'uncf.list'):+.2f} %")

### YOUR TURN 4

Two things make a nonlinear solve hard: a bad starting guess, and a solver that
gives up too early. Try the first.

Our initial head was a flat 660 m everywhere, while the boundaries are 620 and
680. Every interior cell starts in the wrong place, and the transmissivity
computed from that first guess is wrong everywhere too.

Run `unconfined_basin` again with a starting head **closer to the answer** — try
650.0 — and see whether that alone is enough.

- `ok_better_ic` — the success flag
- `buff_better_ic` — the output lines

In [ ]:
# YOUR TURN
ok_better_ic, buff_better_ic, mf_better_ic = ...

In [ ]:
# CHECK
assert ok_better_ic is False, (
    "a better initial condition alone should still not be enough here — "
    f"got success = {ok_better_ic}"
)
print(f"better initial condition: success = {ok_better_ic}")
print("Still failing. A good starting guess helps, but it is not sufficient.")
print("Correct.")

Worth sitting with for a moment: **you improved the model and it still failed.**

Debugging a solver is not a sequence of fixes that each move you closer. It is a
search over several settings at once, and the honest workflow is to change one
thing at a time and keep notes on what you tried.

### The solver settings that matter

`ModflowPcg` has four arguments you will actually adjust:

| Argument | Default | What it does |
|---|---|---|
| `mxiter` | 20 | **outer** iterations — how many times to recompute transmissivity |
| `iter1` | 30 | **inner** iterations per outer one |
| `hclose` | 1e-3 | head-change tolerance for calling it converged |
| `relax` | 1.0 | relaxation — damps oscillation between iterations |

For a nonlinear problem the one that usually matters is **`mxiter`**: with 20
outer iterations there simply are not enough passes to settle a transmissivity
that keeps changing.

In [ ]:
ok_relaxed, buff_relaxed, mf_relaxed = unconfined_basin(
    660.0,
    dict(mxiter=200, iter1=50, hclose=1e-3, rclose=1e2, relax=0.98),
    "uncf_relaxed",
)
print(f"relaxed solver: success = {ok_relaxed}")

head_uncf = flopy.utils.HeadFile(
    str(Path(mf_relaxed.model_ws) / "uncf.hds")
).get_data()
print(f"head {head_uncf.min():.1f} to {head_uncf.max():.1f} m")
print(f"discrepancy {discrepancy(Path(mf_relaxed.model_ws) / 'uncf.list'):+.3f} %")

Ten times the outer iterations, and it converges to a budget that closes.

> **A converged model is not a correct model.** All we have established is that
> the solver found a self-consistent answer. Whether it is the *right* answer is
> a question about the boundary conditions, the conductivity field, and the
> recharge — none of which the solver knows anything about.

### YOUR TURN 5

Compare the confined and unconfined answers for the same physical basin.

`results[250.0]["well_head"]` holds the confined result. Pull the unconfined one
out of `head_uncf`.

- `confined_well_head`
- `unconfined_well_head`
- `difference_m` — unconfined minus confined

In [ ]:
# YOUR TURN
confined_well_head = ...
unconfined_well_head = ...
difference_m = ...

In [ ]:
# CHECK
assert abs(confined_well_head - 654.36) < 0.2, f"got {confined_well_head}"
assert abs(difference_m) > 1.0, (
    "the two should differ by more than a metre — that is the point"
)
print(f"confined   {confined_well_head:8.2f} m")
print(f"unconfined {unconfined_well_head:8.2f} m")
print(f"difference {difference_m:+8.2f} m")
print("Correct.")

The two models differ, and the choice between them is a **statement about the
aquifer**, not a numerical convenience.

- Confined is right when the aquifer is overlain by something that keeps it
  full, and head stays above its top.
- Convertible is right when the water table *is* the top of the saturated zone,
  so pumping thins the aquifer and reduces its ability to deliver water.

The second is more realistic for most of the Tucson basin, and it is why the
model is harder to run.

---

## Part 5 — Dry cells

If head in a convertible cell falls below the layer bottom, the cell has no
saturated thickness left. MODFLOW marks it **dry**, sets its head to `-1e30`, and
takes it out of the solution.

In [ ]:
# Pump hard enough to dewater cells: 8x the rate, and a thinner aquifer
ws = RUN / "week12_dry"
ws.mkdir(parents=True, exist_ok=True)
mf_dry = flopy.modflow.Modflow("dry", model_ws=str(ws), exe_name=str(MF_EXE))
flopy.modflow.ModflowDis(mf_dry, 1, NROW, NCOL, delr=DELR, delc=DELR,
                         top=700.0, botm=640.0, nper=1, steady=True)
ib = np.ones((1, NROW, NCOL), dtype=int)
ib[:, :, 0] = -1
ib[:, :, -1] = -1
strt = np.full((1, NROW, NCOL), 670.0)
strt[:, :, 0] = 660.0
strt[:, :, -1] = 680.0
flopy.modflow.ModflowBas(mf_dry, ibound=ib, strt=strt, hnoflo=-1e30)
flopy.modflow.ModflowLpf(mf_dry, hk=12.0, laytyp=1, ipakcb=53)
flopy.modflow.ModflowWel(mf_dry, stress_period_data={0: [[0, 20, 30, -160000.0]]},
                         ipakcb=53)
flopy.modflow.ModflowRch(mf_dry, rech=1.2e-4, ipakcb=53)
flopy.modflow.ModflowPcg(mf_dry, mxiter=200, iter1=50, hclose=1e-2, rclose=1e2,
                         relax=0.98)
flopy.modflow.ModflowOc(mf_dry, stress_period_data={(0, 0): ["save head", "save budget"]})
mf_dry.write_input()
ok_dry, buff_dry = mf_dry.run_model(silent=True, report=True)

print(f"success = {ok_dry}")
head_dry = flopy.utils.HeadFile(str(ws / "dry.hds")).get_data()
print(f"raw head range: {head_dry.min():.3e} to {head_dry.max():.1f}")

### YOUR TURN 6

That `-1e+30` is not a head. Find the dry cells and handle them.

- `dry_mask` — boolean array, `True` where a cell went dry (head below −1e29)
- `n_dry` — how many
- `head_masked` — a copy of `head_dry[0]` with dry cells set to `np.nan`, so
  plotting and statistics ignore them

In [ ]:
# YOUR TURN
dry_mask = ...
n_dry = ...
head_masked = ...

In [ ]:
# CHECK
assert dry_mask.shape == (NROW, NCOL), f"expected a 2D mask, got {dry_mask.shape}"
assert n_dry > 0, "this model was built to dewater cells — none were found"
assert np.isnan(head_masked[dry_mask]).all(), "dry cells should be NaN"
assert not np.isnan(head_masked[~dry_mask]).any(), "wet cells should keep their values"
print(f"{n_dry} cells went dry ({100 * n_dry / dry_mask.size:.1f}% of the grid)")
print(f"wet-cell head range: {np.nanmin(head_masked):.1f} to "
      f"{np.nanmax(head_masked):.1f} m")
print("Correct.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))

im0 = axes[0].imshow(head_dry[0], cmap="Blues_r",
                     extent=[0, NCOL * DELR / 1000, 0, NROW * DELR / 1000])
fig.colorbar(im0, ax=axes[0], label="head (m)")
axes[0].set_title("Raw output: -1e30 wrecks the colour scale")

im1 = axes[1].imshow(head_masked, cmap="Blues_r",
                     extent=[0, NCOL * DELR / 1000, 0, NROW * DELR / 1000])
axes[1].contourf(np.flipud(dry_mask.astype(float)), levels=[0.5, 1.5],
                 colors=["#AB0520"], alpha=0.8,
                 extent=[0, NCOL * DELR / 1000, 0, NROW * DELR / 1000])
fig.colorbar(im1, ax=axes[1], label="head (m)")
axes[1].set_title("Masked, with dry cells in red")

for ax in axes:
    ax.set_xlabel("distance east (km)")
plt.tight_layout()
plt.show()

**Think about this before next week.** Notice what the left panel does: one
sentinel value destroys the entire figure, exactly as `-999` did to the
water-level series in Week 6. Same lesson, different decade of software.

But the more important point is the red patch. **A dry cell is almost always a
modelling failure rather than a result.** Once a cell is dry it stops conducting
water, so its neighbours dry too, and the failure propagates — MODFLOW-2005 in
particular cannot rewet a dry cell unless you turn on the wetting capability,
which is itself unstable.

If you find dry cells, the honest reading is one of:

- the pumping rate is unphysical
- the layer bottom is too shallow
- the aquifer needed more layers, so the water table can fall within the top one
- you should be using MODFLOW-NWT, which handles dewatering properly

"Report the heads with the dry cells masked out" is not on that list.

---

## Before you leave

1. *Kernel → Restart Kernel and Run All Cells*
2. Fix anything that breaks
3. Save

## What's due

**HW 10 — First MODFLOW model**, Wednesday 11/18 at 11:59pm.

## Next week

Stresses that vary in time: transient simulations, wells that switch on, and
evapotranspiration.

## Stuck?

- This notebook runs eight MODFLOW models. If a cell is slow, that is why.
- `AssertionError` from `confined_basin` means a grid size that doesn't divide
  the domain evenly. Stick to 500, 250, 125.
- Heads of exactly `1e30` are **inactive** (`ibound = 0`); heads of `-1e30` are
  **dry**. Different problems.
- A model that fails with `mxiter=200` usually has a physical problem, not a
  solver one. Check for a boundary head below the layer bottom.
- Office hours: Tuesdays 1:00–2:00pm, Harshbarger 322B.